# PhysAgg-R reproduction

IEEE Latin America Transactions, manuscript ID 11172.

Builds the physical graph, optionally relaunches training with
`python -m src.train.physagg`, and shows the metric tables stored in
`experiments/outputs/physagg_power/`.

To retrain, set `RUN_TRAINING = True` (requires PyTorch Geometric).


In [1]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import pandas as pd

ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Set True to retrain locally.
RUN_TRAINING = False

DATA = ROOT / "data" / "datos_wifi_curated_anon.parquet"
GRAPH_DIR = ROOT / "data" / "graphs" / "physical"
OUT_TEMPORAL = ROOT / "experiments" / "outputs" / "physagg_power" / "temporal_noniso"
OUT_SPATIAL = ROOT / "experiments" / "outputs" / "physagg_power" / "spatial_suchiapa"

print("RUN_TRAINING:", RUN_TRAINING)
print("Data:", DATA.relative_to(ROOT).as_posix(), "| exists:", DATA.exists())

RUN_TRAINING: False
Data: data/datos_wifi_curated_anon.parquet | exists: True


## 1. Build physical graph (same code as CLI)

Equivalent CLI:

```bash
python -m src.graphs.build --input data/datos_wifi_curated_anon.parquet --output data/graphs
```

PhysAgg only needs the **physical** graph; we build that artefact here.

In [2]:
from src.graphs.build import build_physical_graph, save_graph

df = pd.read_parquet(DATA)
nodes, edges = build_physical_graph(df, nearby_threshold_m=30.0, min_co_observations=2)
stats = save_graph(ROOT / "data" / "graphs", "physical", nodes, edges)
print("Graph stats:", {k: stats[k] for k in stats if k != "name"})
print("Saved under:", "data/graphs/physical/")

Graph stats: {'n_nodes': 36010, 'n_edges': 20074, 'n_components': 31315, 'avg_degree': 1.1149125242988058, 'max_degree': 71}
Saved under: data/graphs/physical/


## 2. Training recipes

| Name | Setting |
|:---|:---|
| `temporal_ablation` | Temporal non-isolate, 5 seeds |
| `spatial_holdout` | Municipality hold-out (Suchiapa) |


In [3]:
RECIPES = {
    "temporal_ablation": {
        "output": OUT_TEMPORAL,
        "args": [
            "--run", "all",
            "--seeds", "42,0,1,2,3",
            "--split", "temporal",
            "--subgraph", "non_isolate",
            "--mcnemar", "A2e,A0",
            "--with-rf",
        ],
    },
    "spatial_holdout": {
        "output": OUT_SPATIAL,
        "args": [
            "--run", "all",
            "--seed", "42",
            "--split", "spatial",
            "--spatial-val", "robust",
            "--subgraph", "non_isolate",
            "--mcnemar", "A0,A2e",
        ],
    },
}


def recipe_cli(name: str) -> str:
    """Command with paths relative to the package root."""
    spec = RECIPES[name]
    parts = [
        "python -m src.train.physagg",
        "--curated",
        DATA.relative_to(ROOT).as_posix(),
        "--graph-dir",
        GRAPH_DIR.relative_to(ROOT).as_posix(),
        "--output",
        spec["output"].relative_to(ROOT).as_posix(),
        *spec["args"],
    ]
    return " ".join(parts)


def recipe_command(name: str) -> list[str]:
    spec = RECIPES[name]
    return [
        sys.executable,
        "-m",
        "src.train.physagg",
        "--curated",
        str(DATA),
        "--graph-dir",
        str(GRAPH_DIR),
        "--output",
        str(spec["output"]),
        *spec["args"],
    ]


def run_recipe(name: str) -> Path:
    print("$", recipe_cli(name))
    if not RUN_TRAINING:
        print("Training not launched (RUN_TRAINING = False)")
        return RECIPES[name]["output"]
    RECIPES[name]["output"].mkdir(parents=True, exist_ok=True)
    subprocess.run(recipe_command(name), cwd=ROOT, check=True)
    return RECIPES[name]["output"]


for name in RECIPES:
    print("===", name, "===")
    print(recipe_cli(name))
    print()

=== temporal_ablation ===
python -m src.train.physagg --curated data/datos_wifi_curated_anon.parquet --graph-dir data/graphs/physical --output experiments/outputs/physagg_power/temporal_noniso --run all --seeds 42,0,1,2,3 --split temporal --subgraph non_isolate --mcnemar A2e,A0 --with-rf

=== spatial_holdout ===
python -m src.train.physagg --curated data/datos_wifi_curated_anon.parquet --graph-dir data/graphs/physical --output experiments/outputs/physagg_power/spatial_suchiapa --run all --seed 42 --split spatial --spatial-val robust --subgraph non_isolate --mcnemar A0,A2e



In [4]:
out_temporal = run_recipe("temporal_ablation")
out_spatial = run_recipe("spatial_holdout")


$ python -m src.train.physagg --curated data/datos_wifi_curated_anon.parquet --graph-dir data/graphs/physical --output experiments/outputs/physagg_power/temporal_noniso --run all --seeds 42,0,1,2,3 --split temporal --subgraph non_isolate --mcnemar A2e,A0 --with-rf
Training not launched (RUN_TRAINING = False)
$ python -m src.train.physagg --curated data/datos_wifi_curated_anon.parquet --graph-dir data/graphs/physical --output experiments/outputs/physagg_power/spatial_suchiapa --run all --seed 42 --split spatial --spatial-val robust --subgraph non_isolate --mcnemar A0,A2e
Training not launched (RUN_TRAINING = False)


## 3. Metrics

Tables from `experiments/outputs/physagg_power/`.


In [5]:
def load_summary(out_dir: Path):
    path = out_dir / "summary.json"
    if not path.exists():
        print("Missing:", path.relative_to(ROOT).as_posix())
        return None
    payload = json.loads(path.read_text(encoding="utf-8"))
    rows = payload.get("summary") or []
    return pd.DataFrame(rows)


def load_seed_metrics(out_dir: Path) -> pd.DataFrame:
    rows = []
    for path in sorted(out_dir.glob("metrics_*.json")):
        obj = json.loads(path.read_text(encoding="utf-8"))
        m = obj.get("metrics") or {}
        rows.append(
            {
                "file": path.name,
                "run": obj.get("run"),
                "seed": obj.get("seed"),
                "pr_auc": m.get("pr_auc"),
                "f1": m.get("f1"),
            }
        )
    return pd.DataFrame(rows)


print("Temporal summary")
sum_t = load_summary(out_temporal)
display(sum_t if sum_t is not None else pd.DataFrame({"note": ["run temporal_ablation first"]}))

print("Temporal per-seed metrics")
seed_t = load_seed_metrics(out_temporal)
display(seed_t if not seed_t.empty else pd.DataFrame({"note": ["no metrics_*.json yet"]}))

print("Spatial summary")
sum_s = load_summary(out_spatial)
display(sum_s if sum_s is not None else pd.DataFrame({"note": ["run spatial_holdout first"]}))

Temporal summary


,run,n_seeds,pr_auc_mean,pr_auc_std,f1_mean,f1_std
0,A0,5,0.879505,0.002015,0.765042,0.009481
1,A1,5,0.822877,0.033632,0.726614,0.022251
2,A2,5,0.879572,0.002536,0.762412,0.008147
3,A2e,5,0.885832,0.004678,0.769736,0.010450
4,A3,5,0.879570,0.004823,0.769276,0.007847


Temporal per-seed metrics


,file,run,seed,pr_auc,f1
0,metrics_A0_s0.json,A0,0,0.878081,0.757412
1,metrics_A0_s1.json,A0,1,0.878992,0.775457
2,metrics_A0_s2.json,A0,2,0.883439,0.776750
3,metrics_A0_s3.json,A0,3,0.879023,0.753285
4,metrics_A0_s42.json,A0,42,0.877988,0.762307
5,metrics_A1_s0.json,A1,0,0.841318,0.734807
6,metrics_A1_s1.json,A1,1,0.850833,0.745614
7,metrics_A1_s2.json,A1,2,0.769151,0.689076
8,metrics_A1_s3.json,A1,3,0.855006,0.748886
9,metrics_A1_s42.json,A1,42,0.798078,0.714689


Spatial summary


,run,n_seeds,pr_auc_mean,pr_auc_std,f1_mean,f1_std
0,A0,1,0.627543,0.0,0.460177,0.0
1,A1,1,0.597025,0.0,0.483871,0.0
2,A2,1,0.575776,0.0,0.458333,0.0
3,A2e,1,0.581419,0.0,0.443243,0.0
4,A3,1,0.624665,0.0,0.450593,0.0


## CLI

```bash
python -m src.graphs.build --input data/datos_wifi_curated_anon.parquet --output data/graphs

python -m src.train.physagg \
  --run all --seeds 42,0,1,2,3 --split temporal --subgraph non_isolate \
  --mcnemar A2e,A0 --with-rf \
  --output experiments/outputs/physagg_power/temporal_noniso

python -m src.train.physagg \
  --run all --seed 42 --split spatial --spatial-val robust --subgraph non_isolate \
  --mcnemar A0,A2e \
  --output experiments/outputs/physagg_power/spatial_suchiapa
```
